<a href="https://colab.research.google.com/github/KDTWMU/balancer/blob/main/MMP_BalancerSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title Download the dataset. Run first just once.
import pandas as pd
import requests
from IPython.display import HTML
r = requests.get("https://raw.githubusercontent.com/KDTWMU/kogi2022/main/template.html","r")
html = r.text
try:
  data = pd.read_csv("http://genome.sfu.ca/mmp/mmp_mut_strains_data_Mar14.txt", sep='\t')
except:
  data1 = pd.read_csv("https://raw.githubusercontent.com/KDTWMU/kogi2022/main/db1.csv", header=0)
  data2 = pd.read_csv("https://raw.githubusercontent.com/KDTWMU/kogi2022/main/db2.csv", header=0)
  data = pd.concat([data1,data2])
balancer_db = pd.read_csv("https://raw.githubusercontent.com/KDTWMU/kogi2022/6e11de44c265176ebc43b568e4ea0558a00da3bc/balancer.csv")
balancer_db["Chr"] = balancer_db["Chr"].astype(str).replace('5','X').replace('3','IV').replace('4','V').replace('2','III').replace('1','II').replace('0','I')

In [ ]:
#@title input the gk allele name here (include gk, eg. gk532049)
gk = "gk532049" #@param {type:"string"}
chr = data[data["allele"]==gk]["chr"].values[0]
pos = int(data[data["allele"]==gk]["pos"].values[0])
strain = data[data["allele"]==gk]["strain"].values[0]

def find_gks(balancer, chr, pos):
  start = balancers[balancers["Allele"]==balancer]["Start"].values[0]
  end = balancers[balancers["Allele"]==balancer]["End"].values[0]
  gks = data[(data["chr"]==chr)&(data["pos"]>start)&(data["pos"]<end)&(data["strain"]==strain)]
  nc = data[((data["chr"]==chr)&(data["pos"]<start)&(data["pos"]>end)|(data["chr"]!=chr))&(data["strain"]==strain)]
  return gks, nc

balancers = balancer_db[(balancer_db['Allele'].str.contains("tmC"))&(balancer_db['Chr']==chr)&(balancer_db['Start']<=pos)&(balancer_db['End']>=pos)]
if len(balancers['Allele'].values) >0:
  table_n = 0
  content = "<h1>" +gk + ", "+ chr + ": "+ str(pos) + "</h1>"
  content += "<p>This allele is covered by "+ ", ".join(balancers["Allele"]) +"</p>"
  for balancer in balancers["Allele"]:
    gks, nc = find_gks(balancer, chr, pos)
    content += "<hr>"
    content += f"<h2>The alleles covered by {balancer}</h2><button class=\"toggle-button\" onclick=\"toggleVisibility(\'content{str(table_n)}\')\">Click to toggle visibility</button><div id=\"content{str(table_n)}\" class=\"hidden\">{gks.iloc[:,1:].to_html(index=False)}</div>\n".replace(gk,"<font color=\"red\">"+gk+"</font>")
    table_n += 1
    content += f"<h2>The alleles NOT covered by {balancer}</h2><button class=\"toggle-button\" onclick=\"toggleVisibility(\'content{str(table_n)}\')\">Click to toggle visibility</button><div id=\"content{str(table_n)}\" class=\"hidden\">{nc.iloc[:,1:].to_html(index=False)}</div>\n"
    content += "<hr>"
    table_n += 1
  # Render the HTML template

  rendered_html = html.format(content=content)
  # Save the rendered HTML content to a file
  with open(gk+'.html', 'w') as file:
    file.write(rendered_html)
else:
  print("No tm balancer was found")

display(HTML(rendered_html))